In [15]:
import pandas as pd
from pulp import *
from sklearn.cluster import KMeans
import plotly.graph_objs as go
import pandas as pd
import folium
from branca.colormap import linear

In [16]:
demand = pd.read_csv('../../results/utils/df_demanda_quadrados.csv').drop(['Unnamed: 0'], axis=1)

grids = pd.read_csv('../../results/utils/df_clients_grids.csv').drop(['Unnamed: 0', 'Order Postal Code', 'Latitude', 'Longitude'], axis=1).groupby(by='Grid').mean().reset_index()
grids['lat_centroid'] = (grids['maxLat'] + grids['minLat'])/2
grids['lon_centroid'] = (grids['maxLong'] + grids['minLong'])/2
#grids = grids.drop(['maxLat', 'minLat', 'maxLong', 'minLong'], axis=1)

demand = demand.merge(grids, how='left')

distances = pd.read_csv('../../results/utils/df_dist_dcs.csv').drop(['Unnamed: 0'], axis=1)
distances['grid'] = distances['grid'].apply(int)

candidates = pd.read_excel('../../data/dados.xlsx', sheet_name='candidates')[['Nome', 'Latitude', 'Longitude']]

In [ ]:
# Define number of clusters
num_clusters = [1, 3 , 5, 7]

df_results_all = pd.DataFrame(columns=['n_clusters', 'cluster', 'candidate'])

for num in num_clusters:
    df_grids = grids
    df_grids['n_clusters'] = num

    # Define K-means clustering model
    kmeans = KMeans(n_clusters=num)

    # Extract latitude and longitude columns
    X = df_grids[['lat_centroid', 'lon_centroid']]

    # Cluster the grid cells
    kmeans.fit(X)

    # Add cluster label to each grid cell
    df_grids['cluster'] = kmeans.labels_

    # Iterate over each cluster
    for cluster in range(num):
        cluster_grids = df_grids[df_grids['cluster'] == cluster]
        dist_df = distances[distances['grid'].isin(cluster_grids['Grid'])]
        df = dist_df[['CD', 'Distance']].groupby(by='CD').mean().reset_index()
        candidate = df[df['Distance'] == df['Distance'].min()]['CD'].values[0]
        d = {'n_clusters': [num], 'cluster': [cluster], 'candidate': [candidate]}
        serie = pd.DataFrame(d)
        df_results_all = pd.concat([df_results_all, serie])

    if num == 1:
        # Save result
        df_result = df_grids.merge(df_results_all[['n_clusters', 'cluster', 'candidate']], left_on=['n_clusters', 'cluster'], right_on=['n_clusters', 'cluster'])
    else:
        df_result = pd.concat([df_result, df_grids.merge(df_results_all[['n_clusters', 'cluster', 'candidate']], left_on=['cluster', 'n_clusters'], right_on=['cluster', 'n_clusters'])])


In [23]:
df_result.to_csv('../../results/utils/quadrados_clusterizados_todos_os_clusters.csv')

### Mapa

In [98]:
for num in num_clusters:
    df_result = df_result[df_result['n_clusters'] == num]

    # Create map centered on initial location
    map_ = folium.Map(location=[-23.802923, -46.728143], zoom_start=12)

    # Define color scale
    color_scale = linear.YlOrRd_09.scale(0,num) 

    # Iterate over each row of the dataframe
    for index, row in df_result.iterrows():
        # Extract info from each row
        cell = row['Grid']
        max_lat = row['maxLat']
        min_lat = row['minLat']
        max_long = row['maxLong']
        min_long = row['minLong']
        cluster = row['cluster']
        
        # Compute color based on quantity
        cor = color_scale(cluster)
        
        # Draw rectangle on map for the grid cell with computed color
        folium.Rectangle(
            bounds=[(min_lat, min_long), (max_lat, max_long)],
            color='gray',
            weight=1,
            fill=True,
            fill_color=cor,
            fill_opacity = 0.8, 
            tooltip = row['Grid']
        ).add_to(map_)

    for i in df_result['candidate'].unique():
        candidate = candidates[candidates['Nome'] == i]
        cluster = df_result[df_result['candidate'] == candidate['Nome'].values[0]].iloc[0]
        cor = color_scale(cluster['cluster'])
        folium.Marker(
            location=[candidate['Latitude'], 
            candidate['Longitude']], 
            icon=folium.Icon(color='lightgray', 
            icon='industry', prefix='fa')).add_to(map_)
        folium.CircleMarker(
            location=[candidate['Latitude'], candidate['Longitude']],
            color='black',
            weight=3,
            fill=True,
            fill_color=cor,
            fill_opacity = 0.8, 
            tooltip = row['Grid']
            ).add_to(map_)

    color_scale.caption = 'Cluster'
    color_scale.add_to(map_)

    sw = [df_result['minLat'].min(), df_result['minLong'].min()]
    ne = [df_result['maxLat'].max(), df_result['maxLong'].max()]
    bounds = [sw, ne]
    map_.fit_bounds(bounds=bounds)

    # Save map
    folium.TileLayer('cartodbpositron').add_to(map_)
    map_.save("../../results/maps/" + str(num) + "clusters_cds.html")